# 00｜兩域資料 handoff\n

In [ ]:
from pathlib import Path\nfrom google.colab import drive\ndrive.mount('/content/drive',force_remount=False)\nROOT=Path('/content/drive/MyDrive/姿態判定')\nPROJECT=ROOT/'00_project_config'\nimport sys,yaml\nsys.path.insert(0,str(PROJECT))\nPROTOCOL=yaml.safe_load((PROJECT/'config/protocol.yaml').read_text(encoding='utf-8'))['profiles']['fair_reference']\nassert PROTOCOL=={'batch_size':1,'warmup_images':10,'precision':'fp32','amp':False,'compile':False,'quantization':False,'tensor_rt':False,'rankable_across_models':True}\nprint('✅ fair_reference_v1 已鎖定：',ROOT)\n

In [ ]:
from google.colab import auth\nimport google.auth,json,zipfile\nfrom googleapiclient.discovery import build\nfrom googleapiclient.http import MediaIoBaseDownload\nfrom benchmark_core.data import build_coco_handoff\nauth.authenticate_user();creds,_=google.auth.default();api=build('drive','v3',credentials=creds,cache_discovery=False)\narchive=ROOT/'02_datasets/source_archives';canonical=ROOT/'02_datasets/canonical';archive.mkdir(parents=True,exist_ok=True);canonical.mkdir(parents=True,exist_ok=True)\nsources={'coco_singleperson_val2017':('1dF_y874WDYeMe3Pxz6yzUDcjEhEsAyG6','COCO_SinglePerson_val2017_921.zip','17OLnTkeM5mTzbgJlwvVI4Q04c1vf1SUB','person_keypoints_val2017_singleperson.json',921),'syrip_test100':('1pMnoMJYl9Yniy6cdWXV-4gEhprljTHGG','SyRIP.zip',None,'person_keypoints_validate_infant.json',100)}\ndef get(fid,dst):\n dst=Path(dst);dst.parent.mkdir(parents=True,exist_ok=True)\n if dst.exists() and dst.stat().st_size:return dst\n part=dst.with_suffix(dst.suffix+'.part');part.unlink(missing_ok=True);req=api.files().get_media(fileId=fid,supportsAllDrives=True)\n with part.open('wb') as f:\n  down=MediaIoBaseDownload(f,req,chunksize=8*1024*1024);done=False\n  while not done:\n   status,done=down.next_chunk();print(dst.name,f'{100*status.progress():.1f}%' if status else '')\n part.replace(dst);return dst\ndef unzip(src,dst):\n marker=dst/'.extracted';dst.mkdir(parents=True,exist_ok=True)\n if marker.exists():return\n root=dst.resolve()\n with zipfile.ZipFile(src) as z:\n  assert all((dst*x.filename).resolve().is_relative_to(root) for x in z.infolist()),'unsafe zip'\n  z.extractall(dst)\n marker.write_text('ok')\nfor d,(fid,name,aid,aname,n) in sources.items():\n base=canonical/d;raw=base/'raw';unzip(get(fid,archive/name),raw)\n ann=get(aid,base/'annotations'/aname) if aid else next(iter(raw.rglob(aname)))\n handoff=build_coco_handoff(d,raw,ann,base,n);handoff['mapping_version']='identity_coco17_v1' if d.startswith('coco') else 'syrip_to_coco17_v1';(base/'dataset_manifest.json').write_text(json.dumps(handoff,ensure_ascii=False,indent=2),encoding='utf-8');print('✅',d,handoff)\n